# Day 1 · SciPy

**Block:** SciPy (30 min, including slides and live coding)

Filtering a signal is the job most of you will hand to SciPy first.

1. **Filtering with the FFT** (10 min): after SPL Exercise 42, find the mains
   hum in the LFP's spectrum and remove it
2. **Filtering with `scipy.signal`** (10 min): the same job with a notch
   filter, then a band-pass filter on one channel and on all 384 at once
3. **Stretch:** a smoother spectrum with Welch's method, which distribution
   fits the response times (after SPL Exercise 41), and 2-D minimisation (SPL
   Exercise 40)

We keep using the recording from the NumPy block. The exercises are adapted
from [Scientific Python Lectures, SciPy: high-level scientific
computing](https://lectures.scientific-python.org/intro/scipy/index.html) (CC
BY 4.0), which cleans up a photograph instead.

Before writing your own algorithm, **check whether SciPy already has it**.
The [API reference](https://docs.scipy.org/doc/scipy/reference/) lists every submodule.

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp

with h5py.File("../data/ibl_session.h5") as f:
    rt = f["trials/response_time_s"][:]
    lfp_uv = f["lfp/data"][:] * f["lfp"].attrs["uV_per_count"]
    sampling_rate = f["lfp"].attrs["sampling_rate_Hz"]

trace = lfp_uv[:, 200]
time = np.arange(trace.size) / sampling_rate

## 1. Filtering with the FFT (after SPL Exercise 42)

The recording was made in the United States, where mains electricity
alternates at **60 Hz**. Every cable in the room radiates it, and some ends up
in the LFP. Remove it from `trace` (channel 200, in µV) with the Fast Fourier
Transform:

1. Plot the first 0.5 s of `trace` against `time`.
2. Find the FFT for real signals in `sp.fft` (`rfft`), and the matching
   frequencies (`rfftfreq`, which needs the time between samples). Plot the
   spectrum (the absolute value) against frequency. Is it hard to see
   anything? Why? (Hint: `ax.set_yscale("log")`.)
3. Find the mains hum in the spectrum. Set the components within 1 Hz of
   60 Hz **and of its harmonics**, 120 and 180 Hz, to zero, using a mask on the
   frequencies.
4. Apply the inverse FFT (`irfft`, with `n=` the length of `trace`) to get
   `cleaned`, and plot what you **removed**, `trace - cleaned`, for the first
   0.2 s. What does it look like?

In [ ]:
# Your code here


In [ ]:
# Your code here


*Your answer here.*

In [ ]:
# Your code here


*Your answer here.*

## 2. Filtering with `scipy.signal`

In practice you would design a filter with `sp.signal` and apply it. Filters
come as coefficients: `b, a` for short ones, or `sos` ("second-order
sections"), which is numerically safer for longer ones. `filtfilt` and
`sosfiltfilt` run the filter forwards and then backwards, so the filtered
signal is not delayed relative to the original.

1. Design a 60 Hz notch filter with `b, a = sp.signal.iirnotch(60, Q=30,
   fs=sampling_rate)` and apply it to `trace` with `sp.signal.filtfilt` to
   get `notched`. Plot what it removed, `trace - notched`, for the first
   0.2 s, on the same axes as what the FFT removed. How do they differ?
2. The slow waves of the LFP sit below about 30 Hz. Design a 4th-order
   Butterworth **band-pass** filter from 1 to 30 Hz with
   `sp.signal.butter(4, [1, 30], btype="bandpass", fs=sampling_rate,
   output="sos")`, and apply it with `sp.signal.sosfiltfilt` to get `slow`.
   Plot `trace` and `slow` for the first 2 s.
3. Filter **every channel** of `lfp_uv` at once, with no loop, to get
   `lfp_slow`, shape (5000, 384). (Hint: which `axis` is time?)

In [ ]:
# Your code here


*Your answer here.*

In [ ]:
# Your code here


In [ ]:
# Your code here


## 3. Stretch

### A smoother spectrum: Welch's method

The FFT of the whole recording is noisy: every frequency gets one estimate.
`sp.signal.welch` cuts the signal into overlapping segments, takes the
spectrum of each and averages them. Compute the power spectra of `trace` and
of `notched` with `sp.signal.welch(..., fs=sampling_rate, nperseg=1000)`
(2 s segments, so 0.5 Hz resolution), and plot both on a log scale. Did the
notch remove the 60 Hz peak? What about 120 and 180 Hz?

In [ ]:
# Your code here


In [ ]:
# Your code here


In [ ]:
def sixhump(x):
    return (
        (4 - 2.1 * x[0] ** 2 + x[0] ** 4 / 3) * x[0] ** 2
        + x[0] * x[1]
        + (-4 + 4 * x[1] ** 2) * x[1] ** 2
    )

In [ ]:
# Your code here
